In [ ]:
%matplotlib widget

In [1]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from g4beam import *
from scan import *
from scipy.optimize import differential_evolution
from matplotlib.cm import viridis
import math
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import re
import sys
from pathlib import Path
from matplotlib import cm
import numpy as np
import pandas as pd
from tqdm import *
import pickle
import itertools
import os
from tabulate import tabulate
import tempfile
import glob
import json
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [ ]:
# If particles_after isn't updated with Z = 0.
def convertZ(input_file, output_file):
    event_id_counter = 1
    with open(input_file, "r") as infile, open(output_file, "w") as outfile:
        for line in infile:
            # Skip header lines (those starting with #)
            if line.strip().startswith("#"):
                outfile.write(line)
                continue

            # Split the line into columns
            parts = line.strip().split()
            if len(parts) >= 12:
                parts[2] = "0"  # Set the 3rd column (z) to 0
                # Replace event ID (assuming it's the 9th column, zero-based index 8)
                # Adjust if your event ID is in a different column
                parts[8] = str(event_id_counter)
                event_id_counter += 1
                new_line = " ".join(parts)
                outfile.write(new_line + "\n")
            else:
                # Handle lines that don't match expected format
                outfile.write(line)
    print(f"Updated file saved as '{output_file}'")
    os.remove(input_file)
    return None
convertZ("particles_afterupt.txt", "particles_after.txt")

## Wedge and Dipole

In [3]:
# Make Sure g4bl is here
import os
os.environ["PATH"] += os.pathsep + "/home/incik/G4beamline-3.08/bin"
import shutil
print(shutil.which("g4bl"))

/home/incik/G4beamline-3.08/bin/g4bl


In [6]:
# --------------- Calculation Functions -----------------
def read_g4bl_params(filename):
    """
    Reads 'param' definitions from a .g4bl file and returns a dict of parameter names and values.
    """
    params = {}
    pattern = re.compile(r"param\s+(?:-unset\s+)?(\w+)=([^\s#]+)")
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if not line.startswith("param"):
                continue
            m = pattern.search(line)
            if m:
                key, val = m.groups()
                try:
                    params[key] = float(eval(val, {"__builtins__": None, "pi": math.pi}))
                except Exception:
                    params[key] = val  # keep as string if not numeric
    return params

# Compute wedge elements
def compute_wedge_geometry(params):
    """
    Given parameters like absLEN3, abshgt, abswidth, abshalfangle3, compute geometry info.
    """
    W = params.get("absLEN3")
    H = params.get("abshgt")
    L = params.get("abswidth")
    half_angle_deg = params.get("abshalfangle3")
    offset = params.get("absoffset3", 0)

    if L and half_angle_deg:
        half_angle_rad = math.radians(half_angle_deg)
        L_centerline = W / math.sin(half_angle_rad)

    return {
        "Length_Wedge": L,
        "Height_Base": H,
        "Width_Base": W,
        "Half-angle": half_angle_deg,
        "Centerline_Coords": L_centerline,
        "Offset": offset
    }

# Calculate all will-be-added parameters in the template
def add_missing_params(calcparams_given):
    GAP = 0.1  # in mm (can be up to 1.0 safely)
    A_GAP = 5 # mm
    L_B1 = float(calcparams_given["B1_length"])
    L_D1 = float(calcparams_given["Drift1_length"])
    L_Q1 = float(calcparams_given["Q1_length"])
    L_D2 = float(calcparams_given["Drift2_length"])
    L_B2 = float(calcparams_given["B2_length"])

    wedge_end = geom["Centerline_Coords"]
    B1_z = geom["Centerline_Coords"] + (L_B1/2) + A_GAP
    Drift1_z = B1_z + (L_B1/2) + (L_D1/2) + GAP
    Q1_z     = Drift1_z + (L_D1/2) + (L_Q1/2) + GAP
    Drift2_z = Q1_z + (L_Q1/2) + (L_D2/2)+ GAP
    B2_z     = Drift2_z + (L_D2/2) + (L_B2/2) + GAP
    VD_z     = B2_z + (L_B2/2) + 10.0 + GAP

    B1_end = B1_z + (L_B1/2)
    D1_end = Drift1_z + (L_D1/2)
    Q1_end = Q1_z + (L_Q1/2)
    D2_end = Drift2_z + (L_D2/2)
    B2_end = B2_z + (L_B2/2)

    add_params = {"wedge_end": wedge_end, "B1_z": B1_z, "Drift1_z": Drift1_z, "Q1_z": Q1_z, "Drift2_z":Drift2_z, "B2_z": B2_z, "VD_z": VD_z, 
            "B1_end": B1_end, "B2_end": B2_end,  "D1_end": D1_end, "Q1_end": Q1_end,  "D2_end": D2_end}

    calcparams_given.update(add_params)
    
    for k, v in calcparams_given.items():
        if k == "N_PARTICLES" or k == "VD_FILENAME" or k == "VD_AFILENAME":
            continue
        else:
            calcparams_given[k] = float(v)
    
    return calcparams_given

def check_dipole_clearance(df, B_val, dipole_length_mm, dipole_halfwidth_mm):
    """
    Check which particles clear the dipole aperture based on their Larmor radius
    and deflection geometry.

    Parameters:
      df : pandas.DataFrame with columns ['Px','Py','Pz'] in MeV/c
      B_val : magnetic field (Tesla)
      dipole_length_mm : dipole magnet effective length (mm)
      dipole_halfwidth_mm : half-width of dipole aperture (mm)

    Returns:
      surviving_mask : boolean array, True if particle clears aperture
      r_L_mm : array of Larmor radii (mm)
      delta_r_mm : array of deflections (mm)
    """

    # constants
    e = 1.602176634e-19  # C
    c = 2.99792458e8     # m/s

    # convert MeV/c to SI (kg·m/s)
    mom_perp = np.sqrt(df["Px"]**2 + df["Pz"]**2) * 1e6 * e / c  # B assumed along y

    # Larmor radius [m]
    r_L = mom_perp / (e * np.abs(B_val))

    # convert to mm
    r_L_mm = r_L * 1e3
    s_mm = dipole_length_mm

    # deflection Δr = r * (1 - cos(φ)), φ = s / r
    phi = s_mm / r_L_mm
    delta_r_mm = r_L_mm * (1 - np.cos(phi))

    # check if deflection < half-width
    surviving_mask = delta_r_mm < dipole_halfwidth_mm
    

    return surviving_mask, r_L_mm, delta_r_mm

# Write values to the prepared G4BL template
def write_input_from_template(template_path, out_path, replacements):
    with open(template_path, 'r') as f:
        txt = f.read()
    try:
        txt = txt.format(**replacements)
    except KeyError as e:
        raise RuntimeError(f"Template substitution failed; missing placeholder: {e}")
    with open(out_path, 'w') as f:
        f.write(txt)

#### Wedge Parameters

In [5]:
# ---------------- USER CONFIG ----------------
G4BEAMLINE_CMD = "g4bl"
TEMPLATE_FILE = "G4_FinalCooling_wdc_Template.g4bl"
OUTPUT_DIR = "runs"
VD_AFILENAME = "vd_wedge_end_achromat"
VD_FILENAME = "vd_B1_end_achromat"  # When we sweep, we change this file
N_PARTICLES = 5000                  # increase for lower noise
G4BLFILE = f"G4_FinalCooling_wdc_run"

os.makedirs(OUTPUT_DIR, exist_ok=True)

file_path = "G4_FinalCooling_wdc_Template.g4bl"
params = read_g4bl_params(file_path)
geom = compute_wedge_geometry(params)

print("Parameters found:")
for k, v in params.items():
    print(f"  {k:15s} = {v}")

print("Derived geometry:")
for k, v in geom.items():
    print(f"  {k:20s}: {v}")

Parameters found:
  zbegin          = 0.0
  steppingFormat  = N,GLOBAL,CL,STEP,VOL,PROCESS,P,KE,POLAR,B
  fieldVoxels     = 400,400,400
  maxStep         = 0.5
  minRangeCut     = 1.0
  nparticles      = {N_PARTICLES}
  beamfile        = particles_before.txt
  pi              = 3.141592654
  degrad          = $pi/180
  abshgt          = 10.0
  abswidth        = 100.0
  absLEN3         = 18.0
  abshalfangle3   = 45.0
  absoffset3      = 3.2
  wedge_z         = 0.5*$absLEN3
  VDRad           = 60.0
  wedgeAxis       = 0.0
  noWedge         = 0.0
Derived geometry:
  Length_Wedge        : 100.0
  Height_Base         : 10.0
  Width_Base          : 18.0
  Half-angle          : 45.0
  Centerline_Coords   : 25.455844122715714
  Offset              : 3.2


#### Dipole 1 Parameters

### Sweep Logic:

In [ ]:
# Doing this for B1 and B1_length
B1_field_range = np.arange(-3.0, 3.0, 0.2)
B1_length_range = np.linspace(10, 100, 10) # in mm

# Initialize lists
results = []

for L_val in B1_length_range:
    Dip1 = []
    for i, B_val in enumerate(tqdm(B1_field_range, desc=f"L = {L_val:.1f}")):
        if os.path.exists("field_cell.dat"):
            os.remove("field_cell.dat")

        var_names = ["N_PARTICLES", "B1_field", "B1_width", "B1_height", "B1_length",
                    "Q1_gradient", "Q1_length", "radius_q", "B2_field", "B2_width", "B2_height", "B2_length",
                    "Drift1_width", "Drift1_height", "Drift1_length", "Drift2_width", "Drift2_height", "Drift2_length", "VD_FILENAME", "VD_AFILENAME"]
        #74.7401
        xvec = np.array([
            int(N_PARTICLES), B_val, 30.0, geom["Length_Wedge"], L_val,
            -43.4531, 73.1234, geom["Length_Wedge"], -0.0126, 167.5582, 144.8055, 53.6328,
            120.1356, geom["Length_Wedge"], 463.8105, 102.022, geom["Length_Wedge"], 316.7456,
            "vd_B1_end_achromat.txt", f"./runs/newruns/{VD_AFILENAME}_{L_val:.1f}_{i}.txt"
        ])

        calcparams = {name: val for name, val in zip(var_names, xvec)}
        add_missing_params(calcparams)
        
        write_input_from_template(
            TEMPLATE_FILE,
            f"/home/incik/Cooling_4D/AchromatOneByOne/runs/newruns/{G4BLFILE}_{L_val:.1f}_{i}.g4bl",
            calcparams
        )

        subprocess.run(["g4bl", f"/home/incik/Cooling_4D/AchromatOneByOne/runs/newruns/{G4BLFILE}_{L_val:.1f}_{i}.g4bl"], 
                    capture_output=True, text=True, check=True)

        G4BLOUTPUT = f"/home/incik/Cooling_4D/AchromatOneByOne/runs/newruns/{VD_AFILENAME}_{L_val:.1f}_{i}.txt"
        df = read_trackfile(G4BLOUTPUT)
        
        # Skip if radii exceed Larmor-defined aperture
        mask, r_L_mm, delta_r_mm = check_dipole_clearance(df, B_val, L_val, calcparams["B1_width"]/2)
        
        # If less than 95% of particles clear the aperture, skip this configuration
        if np.mean(mask) < 0.95:
            print(f"Skipping B={B_val:.2f}T, L={L_val:.1f}mm: only {np.mean(mask)*100:.1f}% clear.")
            continue
        
        Dip1.append(B_val)
        
    # store all results for this L_val
    results.append({
        "B1_length": L_val,
        "Dip1": Dip1,
    })
    
    print(results)

